In [0]:
# =============================================================================
# F1-Pulse | Silver Layer — Cleaning & Enrichment
# Notebook: 02_Silver_Transformation
# Author:   Jafar891
# Updated:  2025
#
# Reads Bronze Delta tables, applies schema enforcement, type casting,
# filtering, and enrichment joins. Writes clean data to the Silver layer.
# Idempotent — safe to re-run.
# =============================================================================

import logging
from typing import Optional
from pyspark.sql import DataFrame
from pyspark.sql.functions import (
    col, to_timestamp, current_timestamp, trim, upper, when, lit
)
from pyspark.sql.types import IntegerType, FloatType, BooleanType

# ---------------------------------------------------------------------------
# Configuration
# ---------------------------------------------------------------------------
CATALOG = "f1_project"
BRONZE  = "bronze"
SILVER  = "silver"
YEAR    = 2025

# Minimum lap duration in seconds — filters out obvious bad telemetry
MIN_LAP_DURATION_S = 60.0

# ---------------------------------------------------------------------------
# Logging
# ---------------------------------------------------------------------------
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    datefmt="%Y-%m-%d %H:%M:%S",
)
log = logging.getLogger("f1_pulse.silver")


# ---------------------------------------------------------------------------
# Helpers
# ---------------------------------------------------------------------------

def read_bronze(table_name: str) -> Optional[DataFrame]:
    """Read a Bronze Delta table with existence check."""
    full_table = f"{CATALOG}.{BRONZE}.{table_name}"
    try:
        df = spark.read.table(full_table)  # noqa: F821
        count = df.count()
        log.info(f"  📥 Read {full_table}  ({count:,} rows)")
        if count == 0:
            log.warning(f"  ⚠️  Table {full_table} is empty — downstream results may be incomplete.")
        return df
    except Exception as e:
        log.error(f"  ❌ Failed to read {full_table}: {e}")
        raise


def write_silver(df: DataFrame, table_name: str) -> None:
    """Persist a DataFrame to a Silver Delta table."""
    full_table = f"{CATALOG}.{SILVER}.{table_name}"
    try:
        (
            df.write
              .format("delta")
              .mode("overwrite")
              .option("overwriteSchema", "true")
              .saveAsTable(full_table)
        )
        written = df.count()
        log.info(f"  ✅ Written → {full_table}  ({written:,} rows)")
    except Exception as e:
        log.error(f"  ❌ Failed to write {full_table}: {e}")
        raise


def assert_columns(df: DataFrame, required: list, table_label: str) -> None:
    """Raise clearly if expected columns are missing — catches schema drift early."""
    missing = [c for c in required if c not in df.columns]
    if missing:
        raise ValueError(
            f"Schema drift detected in '{table_label}'. "
            f"Missing columns: {missing}. "
            f"Available: {df.columns}"
        )


def log_quality_check(label: str, total: int, kept: int) -> None:
    dropped = total - kept
    pct = (dropped / total * 100) if total > 0 else 0
    log.info(f"  📊 {label}: {total:,} → {kept:,} rows kept  ({dropped:,} dropped, {pct:.1f}%)")


# ---------------------------------------------------------------------------
# Transformations
# ---------------------------------------------------------------------------

def transform_sessions(bronze_sessions: DataFrame) -> DataFrame:
    """
    Clean and standardise the raw sessions table.
    - Casts timestamps
    - Filters to Race + Qualifying only
    - Renames and selects final columns
    """
    assert_columns(
        bronze_sessions,
        ["date_start", "date_end", "session_type", "session_key",
         "session_name", "location", "country_name", "year"],
        "bronze.raw_sessions"
    )

    total = bronze_sessions.count()

    df = (
        bronze_sessions
        .withColumn("start_time",    to_timestamp(col("date_start")))
        .withColumn("end_time",      to_timestamp(col("date_end")))
        .withColumn("country_name",  trim(col("country_name")))
        .withColumn("session_type",  trim(upper(col("session_type"))))
        .filter(col("session_type").isin("RACE", "QUALIFYING"))
        .filter(col("session_key").isNotNull())
        .filter(col("start_time").isNotNull())
        .select(
            col("session_key").cast(IntegerType()).alias("session_id"),
            trim(col("session_name")).alias("session_name"),
            col("session_type"),
            col("location"),
            col("country_name"),
            col("start_time"),
            col("end_time"),
            col("year").cast(IntegerType()).alias("year"),
            current_timestamp().alias("processed_at"),
            lit(YEAR).cast(IntegerType()).alias("pipeline_year"),
        )
        .dropDuplicates(["session_id"])
    )

    log_quality_check("Sessions", total, df.count())
    return df


def transform_laps(bronze_laps: DataFrame, bronze_drivers: DataFrame) -> DataFrame:
    """
    Enrich laps with driver metadata and apply quality filters.
    - Inner join laps ↔ drivers on driver_number
    - Cast types correctly (no blanket astype(str) pollution from Bronze)
    - Filter out pit-out laps and anomalous lap durations
    - Add data quality flag column for downstream consumers
    """
    assert_columns(
        bronze_laps,
        ["driver_number", "lap_number", "lap_duration", "is_pit_out_lap"],
        "bronze.raw_laps"
    )
    assert_columns(
        bronze_drivers,
        ["driver_number", "full_name", "team_name"],
        "bronze.raw_drivers"
    )

    total_laps = bronze_laps.count()

    # Deduplicate drivers — keep one record per driver_number
    drivers_clean = (
        bronze_drivers
        .select(
            trim(col("driver_number")).alias("driver_number"),
            trim(col("full_name")).alias("full_name"),
            trim(col("team_name")).alias("team_name"),
            col("country_code"),
            col("headshot_url"),
        )
        .dropDuplicates(["driver_number"])
    )

    log.info(f"  👤 Unique drivers in Bronze: {drivers_clean.count()}")

    laps_typed = (
        bronze_laps
        .withColumn("lap_number",    col("lap_number").cast(IntegerType()))
        .withColumn("lap_duration",  col("lap_duration").cast(FloatType()))
        .withColumn("is_pit_out_lap", col("is_pit_out_lap").cast(BooleanType()))
        .withColumn("driver_number", trim(col("driver_number")))
    )

    # Join
    enriched = laps_typed.join(drivers_clean, on="driver_number", how="inner")

    # Quality flag — don't drop bad rows, flag them so Gold can decide
    enriched = enriched.withColumn(
        "is_valid_lap",
        when(col("is_pit_out_lap") == True, False)
        .when(col("lap_duration").isNull(), False)
        .when(col("lap_duration") < MIN_LAP_DURATION_S, False)
        .otherwise(True)
    )

    result = enriched.select(
        col("driver_number"),
        col("full_name"),
        col("team_name"),
        col("country_code"),
        col("headshot_url"),
        col("lap_number"),
        col("lap_duration"),
        col("is_pit_out_lap"),
        col("is_valid_lap"),
        current_timestamp().alias("processed_at"),
    )

    kept = result.count()
    valid = result.filter(col("is_valid_lap") == True).count()
    log_quality_check("Laps (total)", total_laps, kept)
    log.info(f"  ✅ Valid laps (is_valid_lap=True): {valid:,}")
    log.info(f"  ⚠️  Flagged laps (is_valid_lap=False): {kept - valid:,}")

    return result


# ---------------------------------------------------------------------------
# Main
# ---------------------------------------------------------------------------

def run_silver() -> None:
    log.info("=" * 60)
    log.info(f"F1-Pulse Silver Transformation — {YEAR}")
    log.info("=" * 60)

    results = {"success": [], "failed": []}

    # ------------------------------------------------------------------
    # Task 1: Sessions
    # ------------------------------------------------------------------
    log.info("\n[1/2] Transforming sessions …")
    try:
        bronze_sessions = read_bronze(f"raw_sessions_{YEAR}")
        silver_sessions = transform_sessions(bronze_sessions)
        write_silver(silver_sessions, "cleaned_sessions")
        results["success"].append("cleaned_sessions")
    except Exception as e:
        log.error(f"  ❌ Sessions transformation failed: {e}")
        results["failed"].append("cleaned_sessions")

    # ------------------------------------------------------------------
    # Task 2: Enriched laps
    # ------------------------------------------------------------------
    log.info(f"\n[2/2] Enriching laps …")
    try:
        bronze_laps    = read_bronze(f"raw_laps_{YEAR}")
        bronze_drivers = read_bronze(f"raw_drivers_{YEAR}")
        silver_laps    = transform_laps(bronze_laps, bronze_drivers)
        write_silver(silver_laps, "enriched_laps")
        results["success"].append("enriched_laps")
    except Exception as e:
        log.error(f"  ❌ Laps enrichment failed: {e}")
        results["failed"].append("enriched_laps")

    # ------------------------------------------------------------------
    # Summary
    # ------------------------------------------------------------------
    log.info("\n" + "=" * 60)
    log.info("SILVER SUMMARY")
    log.info("=" * 60)
    for t in results["success"]:
        log.info(f"  ✅ {CATALOG}.{SILVER}.{t}")
    for t in results["failed"]:
        log.warning(f"  ❌ {t} — check logs above")
    log.info("=" * 60)

    if results["failed"]:
        raise RuntimeError(f"Silver layer completed with errors: {results['failed']}")


run_silver()

2026-04-08 20:06:05 [INFO] ============================================================
2026-04-08 20:06:05 [INFO] F1-Pulse Silver Transformation — 2025
2026-04-08 20:06:05 [INFO] ============================================================
2026-04-08 20:06:05 [INFO] 
[1/2] Transforming sessions …
2026-04-08 20:06:07 [INFO]   📥 Read f1_project.bronze.raw_sessions_2025  (30 rows)
2026-04-08 20:06:09 [INFO]   📊 Sessions: 30 → 30 rows kept  (0 dropped, 0.0%)
2026-04-08 20:06:13 [INFO]   ✅ Written → f1_project.silver.cleaned_sessions  (30 rows)
2026-04-08 20:06:13 [INFO] 
[2/2] Enriching laps …
2026-04-08 20:06:14 [INFO]   📥 Read f1_project.bronze.raw_laps_2025  (1,156 rows)
2026-04-08 20:06:15 [INFO]   📥 Read f1_project.bronze.raw_drivers_2025  (20 rows)
2026-04-08 20:06:17 [INFO]   👤 Unique drivers in Bronze: 20
2026-04-08 20:06:20 [INFO]   📊 Laps (total): 1,156 → 1,156 rows kept  (0 dropped, 0.0%)
2026-04-08 20:06:20 [INFO]   ✅ Valid laps (is_valid_lap=True): 1,129
2026-04-08 20:06:20 [